## Cell 1: Environment Setup

In [1]:
import os, subprocess

REPO_DIR = "/kaggle/working/Pointnet_Pointnet2_pytorch"
DATA_DIR = f"{REPO_DIR}/data"

# GPU check (safe version)
try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(result.stdout[:500])
except FileNotFoundError:
    print("⚠️ nvidia-smi not found. Please enable GPU in Settings → Accelerator.")

# Install dependencies
os.system("pip install -q h5py tqdm scikit-learn")

# Clone repo
if not os.path.exists(REPO_DIR):
    os.system(f"git clone https://github.com/yanx27/Pointnet_Pointnet2_pytorch {REPO_DIR}")
    print("Repo cloned")
else:
    print("Repo already exists")

os.makedirs(DATA_DIR, exist_ok=True)
os.chdir(REPO_DIR)
print("Setup complete ✅")

Wed Apr 15 09:42:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|       


Cloning into '/kaggle/working/Pointnet_Pointnet2_pytorch'...


Repo cloned
Setup complete ✅


## Cell 2: Data Setup

**Before running:** Add ScanObjectNN `.h5` files as a Kaggle Dataset, then update the paths below.

ModelNet40 will be auto-downloaded from HuggingFace.

In [2]:
import os, h5py, shutil

os.chdir(REPO_DIR)

# ---- ModelNet40: auto-download ----
mn40_dir = f"{DATA_DIR}/modelnet40_ply_hdf5_2048"
if not os.path.exists(mn40_dir) or len(os.listdir(mn40_dir)) < 10:
    print("Downloading ModelNet40...")
    os.makedirs(DATA_DIR, exist_ok=True)
    os.system(f'wget -q "https://huggingface.co/datasets/Msun/modelnet40/resolve/main/modelnet40_ply_hdf5_2048.zip" -O {DATA_DIR}/mn40.zip')
    os.system(f"unzip -o -q {DATA_DIR}/mn40.zip -d {DATA_DIR}/")
    os.system(f"rm {DATA_DIR}/mn40.zip")
    print("ModelNet40 downloaded ✅")
else:
    print("ModelNet40 already exists ✅")

# ---- ScanObjectNN: update paths to your Kaggle dataset ----
SCAN_TRAIN_SRC = "/kaggle/input/datasets/zhizihuadeguojiang/scanobjectnn/test_objectdataset_augmentedrot_scale75.h5"
SCAN_TEST_SRC  = "/kaggle/input/datasets/zhizihuadeguojiang/scanobjectnn/training_objectdataset_augmentedrot_scale75.h5"

os.makedirs(f"{DATA_DIR}/scanobjectnn", exist_ok=True)
if os.path.exists(SCAN_TRAIN_SRC):
    shutil.copy(SCAN_TRAIN_SRC, f"{DATA_DIR}/scanobjectnn/train.h5")
    shutil.copy(SCAN_TEST_SRC,  f"{DATA_DIR}/scanobjectnn/test.h5")
    print("ScanObjectNN copied ✅")
else:
    print("⚠️ ScanObjectNN not found. Update SCAN_TRAIN_SRC / SCAN_TEST_SRC paths above.")

# ---- Verify ----
mn40_files = len(os.listdir(mn40_dir)) if os.path.exists(mn40_dir) else 0
print(f"ModelNet40: {mn40_files} files {'✅' if mn40_files >= 10 else '❌'}")
for name, path in [("train", f"{DATA_DIR}/scanobjectnn/train.h5"),
                   ("test",  f"{DATA_DIR}/scanobjectnn/test.h5")]:
    if os.path.exists(path):
        with h5py.File(path, "r") as f:
            print(f"ScanObjectNN {name}: shape={f['data'].shape} ✅")
    else:
        print(f"ScanObjectNN {name}: NOT FOUND ❌")


ModelNet40 downloaded ✅
ScanObjectNN copied ✅
ModelNet40: 17 files ✅
ScanObjectNN train: shape=(2882, 2048, 3) ✅
ScanObjectNN test: shape=(11416, 2048, 3) ✅


## Cell 3: Code Patches (run once per session)

In [3]:
import re
os.chdir(REPO_DIR)

# Fix 1: HDF5-compatible DataLoader
hdf5_loader = """
import numpy as np, warnings, h5py
from torch.utils.data import Dataset
warnings.filterwarnings("ignore")

def pc_normalize(pc):
    centroid = np.mean(pc, axis=0)
    pc = pc - centroid
    m = np.max(np.sqrt(np.sum(pc**2, axis=1)))
    return pc / m

def load_h5(h5_filename):
    with h5py.File(h5_filename, "r") as f:
        return f["data"][:], f["label"][:]

def load_data(root, split):
    all_data, all_label = [], []
    for line in open("%s/%s_files.txt" % (root, split)).readlines():
        filename = line.strip().split("/")[-1]
        data, label = load_h5("%s/%s" % (root, filename))
        all_data.append(data)
        all_label.append(label)
    return np.concatenate(all_data, axis=0), np.concatenate(all_label, axis=0)

class ModelNetDataLoader(Dataset):
    def __init__(self, root, args, split="train", process_data=False):
        self.npoints = args.num_point
        self.use_normals = args.use_normals
        self.num_category = args.num_category
        self.data, self.label = load_data(root, split)
        print("The size of %s data is %d" % (split, len(self.data)))
    def __len__(self):
        return len(self.data)
    def __getitem__(self, index):
        point_set = self.data[index][:self.npoints, :].astype(np.float32)
        label = self.label[index]
        point_set[:, 0:3] = pc_normalize(point_set[:, 0:3])
        if not self.use_normals:
            point_set = point_set[:, 0:3]
        return point_set, int(label[0])
"""
with open("data_utils/ModelNetDataLoader.py", "w") as f:
    f.write(hdf5_loader)
print("Fix 1: DataLoader replaced \u2705")

# Fix 2: Correct data path
with open("train_classification.py", "r") as f:
    content = f.read()
content = content.replace(
    "data_path = 'data/modelnet40_normal_resampled/'",
    "data_path = 'data/modelnet40_ply_hdf5_2048/'"
)
print("Fix 2: data_path corrected \u2705")

# Fix 3: --augment flag (duplicate-safe)
content = re.sub(r"parser\.add_argument\('--augment'[^\n]*\)\n?", "", content)
content = content.replace(
    "parser.add_argument('--use_normals'",
    "parser.add_argument('--augment', action='store_true', default=False)\n    parser.add_argument('--use_normals'"
)
with open("train_classification.py", "w") as f:
    f.write(content)
count = content.count("--augment")
print(f"Fix 3: --augment count = {count} {chr(9989) if count == 1 else chr(9888)+' check manually'}")

# Fix 4: augment_point_cloud function (duplicate-safe)
with open("provider.py", "r") as f:
    provider = f.read()
if "def augment_point_cloud" not in provider:
    aug_fn = "\ndef augment_point_cloud(batch_data, use_scale=True, use_shift=True, use_rotate=True):\n    augmented = batch_data.copy()\n    if use_scale:\n        augmented = random_scale_point_cloud(augmented)\n    if use_shift:\n        augmented = shift_point_cloud(augmented)\n    if use_rotate:\n        augmented = rotate_point_cloud(augmented)\n    return augmented\n"
    with open("provider.py", "a") as f:
        f.write(aug_fn)
    print("Fix 4: augment_point_cloud added \u2705")
else:
    print("Fix 4: augment_point_cloud already exists \u2705")

# Fix 5: Patch augment switch in training loop (duplicate-safe)
with open("train_classification.py", "r") as f:
    content = f.read()
old = "points = provider.random_scale_point_cloud(points)\n        points = provider.shift_point_cloud(points)"
new = "if args.augment:\n            points = provider.augment_point_cloud(points)\n        else:\n            points = provider.random_scale_point_cloud(points)\n            points = provider.shift_point_cloud(points)"
if old in content:
    content = content.replace(old, new)
    with open("train_classification.py", "w") as f:
        f.write(content)
    print("Fix 5: augment switch patched \u2705")
else:
    print("Fix 5: already patched \u2705")

print("\nAll patches applied \u2705")


Fix 1: DataLoader replaced ✅
Fix 2: data_path corrected ✅
Fix 3: --augment count = 1 ✅
Fix 4: augment_point_cloud added ✅
Fix 5: already patched ✅

All patches applied ✅


## Cell 4: Experiment 1 — PointNet Baseline

Expected: ~90%, Duration: ~1 hour

In [11]:
os.chdir(REPO_DIR)
LOG_EXP1 = "/kaggle/working/exp1_pointnet_baseline"

os.system(f"python train_classification.py "
          f"--model pointnet_cls "
          f"--log_dir {LOG_EXP1} "
          f"--epoch 200 --batch_size 32 "
          f"--learning_rate 0.001")

print("Exp1 complete ✅")


PARAMETER ...
Namespace(use_cpu=False, gpu='0', batch_size=32, model='pointnet_cls', num_category=40, epoch=200, learning_rate=0.001, num_point=1024, optimizer='Adam', log_dir='/kaggle/working/exp1_pointnet_baseline', decay_rate=0.0001, augment=False, use_normals=False, process_data=False, use_uniform_sample=False)
Load dataset ...
The size of train data is 9840
The size of test data is 2468
No existing model, starting training from scratch...
Epoch 1 (1/200):


100%|██████████| 307/307 [00:17<00:00, 17.33it/s]


Train Instance Accuracy: 0.425794


100%|██████████| 78/78 [00:02<00:00, 35.18it/s]


Test Instance Accuracy: 0.450321, Class Accuracy: 0.326229
Best Instance Accuracy: 0.450321, Class Accuracy: 0.326229
Saving at /kaggle/working/exp1_pointnet_baseline/checkpoints/best_model.pth
Epoch 2 (2/200):


...

Test Instance Accuracy: 0.891426, Class Accuracy: 0.853206
Best Instance Accuracy: 0.900641, Class Accuracy: 0.870333
Epoch 200 (200/200):


100%|██████████| 307/307 [00:18<00:00, 16.39it/s]


Train Instance Accuracy: 0.974450


100%|██████████| 78/78 [00:02<00:00, 34.66it/s]


Test Instance Accuracy: 0.889824, Class Accuracy: 0.852987
Best Instance Accuracy: 0.900641, Class Accuracy: 0.870333
Exp1 complete ✅


## Cell 5: Experiment 2 — PointNet++ Baseline

Expected: ~90.5%, Duration: ~3 hours

In [8]:
os.chdir(REPO_DIR)
LOG_EXP2 = "/kaggle/working/exp2_pointnet2_baseline"

os.system(f"python train_classification.py "
          f"--model pointnet2_cls_msg "
          f"--log_dir {LOG_EXP2} "
          f"--epoch 40 --batch_size 24 "
          f"--learning_rate 0.001")

print("Exp2 complete ✅")


PARAMETER ...
Namespace(use_cpu=False, gpu='0', batch_size=24, model='pointnet2_cls_msg', num_category=40, epoch=40, learning_rate=0.001, num_point=1024, optimizer='Adam', log_dir='/kaggle/working/exp2_pointnet2_baseline', decay_rate=0.0001, augment=False, use_normals=False, process_data=False, use_uniform_sample=False)
Load dataset ...
The size of train data is 9840
The size of test data is 2468
No existing model, starting training from scratch...
Epoch 1 (1/40):


  0%|          | 0/103 [00:00<?, ?it/s]

Train Instance Accuracy: 0.580488


100%|██████████| 103/103 [00:42<00:00,  2.40it/s]


Test Instance Accuracy: 0.672735, Class Accuracy: 0.578137
Best Instance Accuracy: 0.672735, Class Accuracy: 0.578137
Saving at /kaggle/working/exp2_pointnet2_baseline/checkpoints/best_model.pth
Epoch 2 (2/40):


...

Test Instance Accuracy: 0.903641, Class Accuracy: 0.868187
Best Instance Accuracy: 0.914887, Class Accuracy: 0.879869
Epoch 40 (40/40):


100%|██████████| 410/410 [06:13<00:00,  1.10it/s]


Train Instance Accuracy: 0.919512


100%|██████████| 103/103 [00:42<00:00,  2.41it/s]


Test Instance Accuracy: 0.904450, Class Accuracy: 0.873486
Best Instance Accuracy: 0.914887, Class Accuracy: 0.879869
Exp2 complete ✅


## Cell 6: Experiment 3 — PointNet++ with Enhanced Augmentation

Expected: ~92.3–92.5%, Duration: ~3 hours

In [4]:
os.chdir(REPO_DIR)
LOG_EXP3 = "/kaggle/working/exp3_pointnet2_augmented"

os.system(f"python train_classification.py "
          f"--model pointnet2_cls_msg "
          f"--log_dir {LOG_EXP3} "
          f"--epoch 40 --batch_size 24 "
          f"--learning_rate 0.001 "
          f"--augment")

print("Exp3 complete ✅")


PARAMETER ...
Namespace(use_cpu=False, gpu='0', batch_size=24, model='pointnet2_cls_msg', num_category=40, epoch=40, learning_rate=0.001, num_point=1024, optimizer='Adam', log_dir='/kaggle/working/exp3_pointnet2_augmented', decay_rate=0.0001, augment=True, use_normals=False, process_data=False, use_uniform_sample=False)
Load dataset ...
The size of train data is 9840
The size of test data is 2468
No existing model, starting training from scratch...
Epoch 1 (1/40):


  0%|          | 0/103 [00:00<?, ?it/s]

Train Instance Accuracy: 0.578150


100%|██████████| 103/103 [00:42<00:00,  2.40it/s]


Test Instance Accuracy: 0.644094, Class Accuracy: 0.543508
Best Instance Accuracy: 0.644094, Class Accuracy: 0.543508
Saving at /kaggle/working/exp3_pointnet2_augmented/checkpoints/best_model.pth
Epoch 2 (2/40):


...

Test Instance Accuracy: 0.909709, Class Accuracy: 0.877451
Best Instance Accuracy: 0.912136, Class Accuracy: 0.877451
Epoch 40 (40/40):


100%|██████████| 410/410 [06:14<00:00,  1.10it/s]


Train Instance Accuracy: 0.922663


100%|██████████| 103/103 [00:42<00:00,  2.42it/s]


Test Instance Accuracy: 0.919013, Class Accuracy: 0.891862
Best Instance Accuracy: 0.919013, Class Accuracy: 0.891862
Saving at /kaggle/working/exp3_pointnet2_augmented/checkpoints/best_model.pth
Exp3 complete ✅


In [13]:
import shutil

# 打包 Exp3
shutil.make_archive("/kaggle/working/exp1_results", "zip",
                    "/kaggle/working/exp1_pointnet_baseline")
print("exp2_results.zip ready ✅")

exp2_results.zip ready ✅


In [7]:
with open("/kaggle/working/exp3_pointnet2_augmented/logs/pointnet2_cls_msg.txt") as f:
    lines = f.readlines()
# 打印最后20行
print("".join(lines[-20:]))

2026-04-15 13:53:39,199 - Model - INFO - Best Instance Accuracy: 0.912136, Class Accuracy: 0.869367
2026-04-15 13:53:39,199 - Model - INFO - Epoch 37 (37/40):
2026-04-15 13:59:53,610 - Model - INFO - Train Instance Accuracy: 0.902744
2026-04-15 14:00:36,519 - Model - INFO - Test Instance Accuracy: 0.892314, Class Accuracy: 0.870408
2026-04-15 14:00:36,519 - Model - INFO - Best Instance Accuracy: 0.912136, Class Accuracy: 0.870408
2026-04-15 14:00:36,519 - Model - INFO - Epoch 38 (38/40):
2026-04-15 14:06:50,737 - Model - INFO - Train Instance Accuracy: 0.912297
2026-04-15 14:07:33,576 - Model - INFO - Test Instance Accuracy: 0.898706, Class Accuracy: 0.865450
2026-04-15 14:07:33,576 - Model - INFO - Best Instance Accuracy: 0.912136, Class Accuracy: 0.870408
2026-04-15 14:07:33,576 - Model - INFO - Epoch 39 (39/40):
2026-04-15 14:13:47,908 - Model - INFO - Train Instance Accuracy: 0.909959
2026-04-15 14:14:30,797 - Model - INFO - Test Instance Accuracy: 0.909709, Class Accuracy: 0.87745

## Cell 7: Experiment 4 — Generalization Test on ScanObjectNN

In [22]:
import os, sys, h5py, numpy as np, torch
from torch.utils.data import Dataset, DataLoader

os.chdir('/kaggle/working/Pointnet_Pointnet2_pytorch')
sys.path.insert(0, '/kaggle/working/Pointnet_Pointnet2_pytorch')
sys.path.insert(0, '/kaggle/working/Pointnet_Pointnet2_pytorch/models')

# ══════════════════════════════════════════════════════════
# ScanObjectNN DataLoader
# ══════════════════════════════════════════════════════════
def pc_normalize(pc):
    centroid = np.mean(pc, axis=0)
    pc = pc - centroid
    m = np.max(np.sqrt(np.sum(pc**2, axis=1)))
    return pc / m

class ScanObjectNNDataset(Dataset):
    def __init__(self, h5_path, num_points=1024):
        with h5py.File(h5_path, 'r') as f:
            self.data  = f['data'][:].astype(np.float32)
            self.label = f['label'][:].astype(np.int64).reshape(-1)
        self.num_points = num_points
        print(f"Loaded {len(self.data)} samples, "
              f"label range: {self.label.min()}–{self.label.max()}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        pts = self.data[idx, :self.num_points, :3]
        pts = pc_normalize(pts)
        return torch.tensor(pts, dtype=torch.float32), self.label[idx]

# ══════════════════════════════════════════════════════════
# Build extended ScanObjectNN → ModelNet40 label mapping
# ══════════════════════════════════════════════════════════
SCAN_CLASSES = ['bag','bin','box','cabinet','chair',
                'desk','display','door','shelf','table',
                'bed','pillow','sink','sofa','toilet']

SCAN_TO_MN40 = {
    'bag':     None,          # not in ModelNet40
    'bin':     None,          # not in ModelNet40
    'box':     None,          # not in ModelNet40
    'cabinet': 'wardrobe',    # closest semantic match
    'chair':   'chair',
    'desk':    'desk',
    'display': 'monitor',     # alias
    'door':    'door',
    'shelf':   'bookshelf',   # alias
    'table':   'table',
    'bed':     'bed',
    'pillow':  None,          # not in ModelNet40
    'sink':    'sink',
    'sofa':    'sofa',
    'toilet':  'toilet',
}

with open('data/modelnet40_ply_hdf5_2048/shape_names.txt') as f:
    MN40_CLASSES = [l.strip() for l in f.readlines()]

label_map = {}
for scan_idx, scan_name in enumerate(SCAN_CLASSES):
    mn40_name = SCAN_TO_MN40.get(scan_name)
    if mn40_name and mn40_name in MN40_CLASSES:
        label_map[scan_idx] = MN40_CLASSES.index(mn40_name)
    else:
        label_map[scan_idx] = -1

print("Class mapping (ScanObjectNN → ModelNet40):")
for s, m in label_map.items():
    mn_name = MN40_CLASSES[m] if m >= 0 else '*** NO MATCH ***'
    print(f"  [{s:2d}] {SCAN_CLASSES[s]:12s} → [{m:2d}] {mn_name}")

matched = sum(1 for v in label_map.values() if v >= 0)
print(f"\n{matched}/15 classes matched ✅")

# ══════════════════════════════════════════════════════════
# Evaluation with label mapping
# ══════════════════════════════════════════════════════════
def evaluate_with_mapping(model, loader, device, label_map):
    model.eval()
    correct = total = 0
    class_correct = np.zeros(15)
    class_total   = np.zeros(15)

    with torch.no_grad():
        for pts, labels in loader:
            pts    = pts.to(device).transpose(2, 1)
            logits, _ = model(pts)
            preds  = logits.argmax(dim=1).cpu().numpy()
            labels = labels.numpy()

            for p, l in zip(preds, labels):
                mn40_label = label_map.get(int(l), -1)
                if mn40_label < 0:
                    continue
                total += 1
                class_total[l] += 1
                if p == mn40_label:
                    correct += 1
                    class_correct[l] += 1

    inst_acc  = correct / total if total > 0 else 0
    valid     = class_total > 0
    class_acc = np.mean(class_correct[valid] / class_total[valid])
    return inst_acc, class_acc

# ══════════════════════════════════════════════════════════
# Load ScanObjectNN test set
# ══════════════════════════════════════════════════════════
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

scan_dataset = ScanObjectNNDataset('data/scanobjectnn/test.h5', num_points=1024)
scan_loader  = DataLoader(scan_dataset, batch_size=24, shuffle=False, num_workers=2)
print(f"\nScanObjectNN test set ready ✅  (device: {device})")

# ══════════════════════════════════════════════════════════
# Test Exp2 — PointNet++ (ModelNet40 → ScanObjectNN)
# ══════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  Exp2 — PointNet++ | ModelNet40 → ScanObjectNN")
print("="*55)
from models.pointnet2_cls_msg import get_model as get_pn2

model_exp2 = get_pn2(40, normal_channel=False).to(device)
ckpt = torch.load(
    '/kaggle/working/exp2_pointnet2_baseline/checkpoints/best_model.pth',
    map_location=device, weights_only=False)
model_exp2.load_state_dict(ckpt['model_state_dict'])

inst2, cls2 = evaluate_with_mapping(model_exp2, scan_loader, device, label_map)
print(f"  Instance Accuracy : {inst2*100:.2f}%")
print(f"  Class    Accuracy : {cls2*100:.2f}%")

# ══════════════════════════════════════════════════════════
# Test Exp1 — PointNet (ModelNet40 → ScanObjectNN)
# ══════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  Exp1 — PointNet   | ModelNet40 → ScanObjectNN")
print("="*55)
from models.pointnet_cls import get_model as get_pn1

model_exp1 = get_pn1(40, normal_channel=False).to(device)
ckpt = torch.load(
    '/kaggle/working/exp1_pointnet_baseline/checkpoints/best_model.pth',
    map_location=device, weights_only=False)
model_exp1.load_state_dict(ckpt['model_state_dict'])

inst1, cls1 = evaluate_with_mapping(model_exp1, scan_loader, device, label_map)
print(f"  Instance Accuracy : {inst1*100:.2f}%")
print(f"  Class    Accuracy : {cls1*100:.2f}%")

# ══════════════════════════════════════════════════════════
# Domain Gap Summary
# ══════════════════════════════════════════════════════════
print("\n" + "="*55)
print("  Domain Gap Summary (ModelNet40 → ScanObjectNN)")
print(f"  Evaluated on {matched}/15 matched classes")
print("="*55)
print(f"  {'Model':<22} {'Instance Acc':>13} {'Class Acc':>11}")
print(f"  {'-'*48}")
print(f"  {'Exp1  PointNet':<22} {inst1*100:>12.2f}% {cls1*100:>10.2f}%")
print(f"  {'Exp2  PointNet++':<22} {inst2*100:>12.2f}% {cls2*100:>10.2f}%")
mn40_exp1 = 90.06
mn40_exp2 = 91.49   # update when Exp2 training done
print(f"\n  ModelNet40 Best Accuracy (reference):")
print(f"  {'Exp1  PointNet':<22} {mn40_exp1:>12.2f}%")
print(f"  {'Exp2  PointNet++':<22} {mn40_exp2:>12.2f}%")
print(f"\n  Accuracy drop confirms synthetic-to-real domain gap.")
print("\nGeneralization test complete ✅")

Class mapping (ScanObjectNN → ModelNet40):
  [ 0] bag          → [-1] *** NO MATCH ***
  [ 1] bin          → [-1] *** NO MATCH ***
  [ 2] box          → [-1] *** NO MATCH ***
  [ 3] cabinet      → [38] wardrobe
  [ 4] chair        → [ 8] chair
  [ 5] desk         → [12] desk
  [ 6] display      → [22] monitor
  [ 7] door         → [13] door
  [ 8] shelf        → [ 4] bookshelf
  [ 9] table        → [33] table
  [10] bed          → [ 2] bed
  [11] pillow       → [-1] *** NO MATCH ***
  [12] sink         → [29] sink
  [13] sofa         → [30] sofa
  [14] toilet       → [35] toilet

11/15 classes matched ✅
Loaded 11416 samples, label range: 0–14

ScanObjectNN test set ready ✅  (device: cuda)

  Exp2 — PointNet++ | ModelNet40 → ScanObjectNN
  Instance Accuracy : 19.09%
  Class    Accuracy : 17.66%

  Exp1 — PointNet   | ModelNet40 → ScanObjectNN
  Instance Accuracy : 12.70%
  Class    Accuracy : 12.14%

  Domain Gap Summary (ModelNet40 → ScanObjectNN)
  Evaluated on 11/15 matched classes
 

## Cell 8: Save Results

All files in `/kaggle/working/` are automatically available in the **Output** tab.
Run this cell to verify everything is saved.

In [12]:
import os

for exp in ["exp1_pointnet_baseline", "exp2_pointnet2_baseline", "exp3_pointnet2_augmented"]:
    ckpt = f"/kaggle/working/{exp}/checkpoints/best_model.pth"
    log  = f"/kaggle/working/{exp}/logs/"
    ckpt_ok = "✅" if os.path.exists(ckpt) else "❌ NOT FOUND"
    log_ok  = "✅" if os.path.exists(log) else "❌ NOT FOUND"
    print(f"{exp}:")
    print(f"  checkpoint: {ckpt_ok}")
    print(f"  logs:       {log_ok}")

print("\nDownload all results from Kaggle Output tab ✅")


exp1_pointnet_baseline:
  checkpoint: ✅
  logs:       ✅
exp2_pointnet2_baseline:
  checkpoint: ✅
  logs:       ✅
exp3_pointnet2_augmented:
  checkpoint: ✅
  logs:       ✅

Download all results from Kaggle Output tab ✅


## Cell 8: 结果图表可视化

In [26]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import re, os

os.makedirs('/kaggle/working/figures', exist_ok=True)

# ══════════════════════════════════════════════════════════
# 1. Parse training log
# ══════════════════════════════════════════════════════════
def parse_log(log_path):
    epochs, inst_acc, cls_acc = [], [], []
    cur_ep = 0
    with open(log_path) as f:
        lines = f.readlines()
    for line in lines:
        m_ep = re.search(r'Epoch \d+ \((\d+)/\d+\)', line)
        if m_ep:
            cur_ep = int(m_ep.group(1))
        m_test = re.search(r'Test Instance Accuracy: ([0-9.]+), Class Accuracy: ([0-9.]+)', line)
        if m_test:
            epochs.append(cur_ep)
            inst_acc.append(float(m_test.group(1)) * 100)
            cls_acc.append(float(m_test.group(2)) * 100)
    return epochs, inst_acc, cls_acc

LOG_EXP2 = '/kaggle/working/exp2_pointnet2_baseline/logs/pointnet2_cls_msg.txt'
LOG_EXP3 = '/kaggle/working/exp3_pointnet2_augmented/logs/pointnet2_cls_msg.txt'

# ══════════════════════════════════════════════════════════
# 2. Training curves: Exp2 vs Exp3
# ══════════════════════════════════════════════════════════
ep2, ia2, ca2 = parse_log(LOG_EXP2)
ep3, ia3, ca3 = parse_log(LOG_EXP3)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(ep2, ia2, label='Exp2: PointNet++ Baseline', color='#2196F3', linewidth=1.5)
axes[0].plot(ep3, ia3, label='Exp3: PointNet++ + Augment', color='#FF5722', linewidth=1.5)
axes[0].axhline(y=91.9, color='gray', linestyle='--', linewidth=1,
                alpha=0.7, label='PointNet++ paper (91.9%)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Instance Accuracy (%)')
axes[0].set_title('Test Instance Accuracy over Epochs')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep2, ca2, label='Exp2: PointNet++ Baseline', color='#2196F3', linewidth=1.5)
axes[1].plot(ep3, ca3, label='Exp3: PointNet++ + Augment', color='#FF5722', linewidth=1.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Class Accuracy (%)')
axes[1].set_title('Test Class Accuracy over Epochs')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print("Training curves saved ✅")

# ══════════════════════════════════════════════════════════
# 3. Bar chart: Your results vs paper reported
# ══════════════════════════════════════════════════════════

# ── Your results (update Exp2 & Exp3 when training complete) ──
your_inst = [90.06, 91.49, 91.90]   # Exp1 / Exp2 / Exp3
your_cls  = [87.03, 87.99, 89.19]   # Exp1 / Exp2 / Exp3

# ── Paper reported results ──
# PointNet (Qi et al. 2017): 89.2% on ModelNet40
# PointNet++ MSG (Qi et al. 2017): 91.9% on ModelNet40
paper_inst = [89.2, 91.9, 91.9]    # Exp3 has no paper baseline (augment is custom)
paper_cls  = [86.2, None, None]     # PointNet paper class acc; PointNet++ not reported

exp_labels = ['Exp1\nPointNet\nBaseline',
              'Exp2\nPointNet++\nBaseline',
              'Exp3\nPointNet++\n+Augment']

x     = np.arange(len(exp_labels))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))

bars_yours = ax.bar(x - width/2, your_inst, width,
                    label='Your Result (Instance Acc)', color='#2196F3')
bars_paper = ax.bar(x + width/2, paper_inst, width,
                    label='Paper Reported (Instance Acc)', color='#90CAF9',
                    edgecolor='#2196F3', linewidth=1.2)

# Value labels
for bar in list(bars_yours) + list(bars_paper):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.08,
            f'{h:.2f}%', ha='center', va='bottom', fontsize=8)

# Annotate Exp3 has no paper baseline
ax.text(x[2] + width/2, paper_inst[2] + 0.5,
        '(same arch)', ha='center', fontsize=7, color='gray')

ax.set_ylim(85, 95)
ax.set_xticks(x)
ax.set_xticklabels(exp_labels)
ax.set_ylabel('Instance Accuracy (%)')
ax.set_title('ModelNet40 Classification — Your Results vs Paper Reported')
ax.legend(loc='lower right')
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/figures/accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("Accuracy comparison chart saved ✅")

# ══════════════════════════════════════════════════════════
# 4. Print summary table
# ══════════════════════════════════════════════════════════
print("\n" + "="*62)
print("  ModelNet40 Results Summary")
print("="*62)
print(f"  {'Experiment':<28} {'Yours':>8} {'Paper':>8} {'Gap':>8}")
print(f"  {'-'*58}")
for i, (label, y, p) in enumerate(zip(
        ['Exp1 PointNet', 'Exp2 PointNet++', 'Exp3 PointNet++ +Aug'],
        your_inst, paper_inst)):
    gap = y - p
    print(f"  {label:<28} {y:>7.2f}% {p:>7.2f}% {gap:>+7.2f}%")

print(f"\n  * Exp1 slightly exceeds paper due to HDF5 data split difference.")
print(f"  * Exp3 uses custom augmentation; paper result is same architecture.")
print("="*62)
print("\nAll Cell 8 outputs saved ✅")
print("Files:", os.listdir('/kaggle/working/figures/'))

Training curves saved ✅
Accuracy comparison chart saved ✅

  ModelNet40 Results Summary
  Experiment                      Yours    Paper      Gap
  ----------------------------------------------------------
  Exp1 PointNet                  90.06%   89.20%   +0.86%
  Exp2 PointNet++                91.49%   91.90%   -0.41%
  Exp3 PointNet++ +Aug           91.90%   91.90%   +0.00%

  * Exp1 slightly exceeds paper due to HDF5 data split difference.
  * Exp3 uses custom augmentation; paper result is same architecture.

All Cell 8 outputs saved ✅
Files: ['failure_analysis.txt', 'training_curves.png', 'failure_cases.png', 'accuracy_comparison.png', 'pointcloud_samples.png']


## Cell 9:  Point Cloud Visualization & Failure Cases

In [25]:
import h5py, numpy as np, os, sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import torch

sys.path.insert(0, '/kaggle/working/Pointnet_Pointnet2_pytorch')
sys.path.insert(0, '/kaggle/working/Pointnet_Pointnet2_pytorch/models')
os.makedirs('/kaggle/working/figures', exist_ok=True)

# ── 1. Load test data ──
with h5py.File('/kaggle/working/Pointnet_Pointnet2_pytorch/data/'
               'modelnet40_ply_hdf5_2048/ply_data_test0.h5', 'r') as f:
    test_data   = f['data'][:]
    test_labels = f['label'][:]

shapenames = [l.strip() for l in open(
    '/kaggle/working/Pointnet_Pointnet2_pytorch/data/'
    'modelnet40_ply_hdf5_2048/shape_names.txt')]

def plot_pc(ax, pts, title, color='steelblue'):
    ax.scatter(pts[:, 0], pts[:, 2], pts[:, 1], s=0.8, c=color, alpha=0.7)
    ax.set_title(title, fontsize=9)
    ax.set_axis_off()
    ax.set_box_aspect([1, 1, 1])

# ── 2. Visualize 6 representative classes ──
target_classes = [0, 4, 8, 12, 20, 30]
fig = plt.figure(figsize=(16, 3))
for i, cls in enumerate(target_classes):
    idx = np.where(test_labels[:, 0] == cls)[0][0]
    ax  = fig.add_subplot(1, 6, i + 1, projection='3d')
    plot_pc(ax, test_data[idx], shapenames[cls])
plt.suptitle('ModelNet40 — Point Cloud Samples (6 Categories)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/pointcloud_samples.png', dpi=150, bbox_inches='tight')
plt.close()
print("Point cloud samples saved ✅")

# ── 3. Load Exp3 best model ──
from models.pointnet2_cls_msg import get_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = get_model(40, normal_channel=False).to(device)
ckpt   = torch.load(
    '/kaggle/working/exp3_pointnet2_augmented/checkpoints/best_model.pth',
    map_location=device, weights_only=False)  # ← fix
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print("Exp3 model loaded ✅")

# ── 4. Collect failure cases (first 500 test samples) ──
fail_cases = []
with torch.no_grad():
    for i in range(min(500, len(test_data))):
        pts  = torch.tensor(test_data[i:i+1, :1024, :3],
                            dtype=torch.float32).to(device)
        pts  = pts.transpose(2, 1)
        logits, _ = model(pts)
        pred = logits.argmax(dim=1).item()
        gt   = int(test_labels[i][0])
        if pred != gt:
            fail_cases.append((i, gt, pred))

print(f"Failure cases found: {len(fail_cases)} / 500  "
      f"(error rate: {len(fail_cases)/500*100:.1f}%)")

# ── 5. Visualize top 6 failure cases ──
n_show = min(6, len(fail_cases))
fig = plt.figure(figsize=(16, 3))
for i, (idx, gt, pred) in enumerate(fail_cases[:n_show]):
    ax = fig.add_subplot(1, n_show, i + 1, projection='3d')
    plot_pc(ax, test_data[idx],
            f'GT: {shapenames[gt]}\nPred: {shapenames[pred]}',
            color='tomato')
plt.suptitle('Failure Cases — Exp3 PointNet++ (Ground Truth → Wrong Prediction)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/failure_cases.png', dpi=150, bbox_inches='tight')
plt.close()
print("Failure cases saved ✅")

# ── 6. Save failure statistics ──
from collections import Counter
fail_gts   = Counter([shapenames[gt]   for _, gt, _   in fail_cases])
fail_preds = Counter([shapenames[pred] for _, _,  pred in fail_cases])

with open('/kaggle/working/figures/failure_analysis.txt', 'w') as f:
    f.write("=== Failure Case Analysis (Exp3 PointNet++, first 500 test samples) ===\n\n")
    f.write(f"Total failures : {len(fail_cases)}\n")
    f.write(f"Error rate     : {len(fail_cases)/500*100:.1f}%\n\n")
    f.write("Most misclassified ground-truth classes:\n")
    for cls, cnt in fail_gts.most_common(10):
        f.write(f"  {cls:20s}: {cnt} errors\n")
    f.write("\nMost common wrong predictions:\n")
    for cls, cnt in fail_preds.most_common(10):
        f.write(f"  {cls:20s}: {cnt} times\n")

print("Failure analysis report saved ✅")
print("All output files:", os.listdir('/kaggle/working/figures/'))

Point cloud samples saved ✅
Exp3 model loaded ✅
Failure cases found: 51 / 500  (error rate: 10.2%)
Failure cases saved ✅
Failure analysis report saved ✅
All output files: ['failure_analysis.txt', 'training_curves.png', 'failure_cases.png', 'accuracy_comparison.png', 'pointcloud_samples.png']


In [27]:
import shutil, os

os.makedirs('/kaggle/working/submission', exist_ok=True)

# ── Pack checkpoints ──
for exp in ['exp1_pointnet_baseline',
            'exp2_pointnet2_baseline',
            'exp3_pointnet2_augmented']:
    ckpt_src = f'/kaggle/working/{exp}/checkpoints/best_model.pth'
    log_src  = f'/kaggle/working/{exp}/logs'
    if os.path.exists(ckpt_src):
        os.makedirs(f'/kaggle/working/submission/{exp}', exist_ok=True)
        shutil.copy(ckpt_src, f'/kaggle/working/submission/{exp}/best_model.pth')
    if os.path.exists(log_src):
        shutil.copytree(log_src,
                        f'/kaggle/working/submission/{exp}/logs',
                        dirs_exist_ok=True)

# ── Pack figures ──
if os.path.exists('/kaggle/working/figures'):
    shutil.copytree('/kaggle/working/figures',
                    '/kaggle/working/submission/figures',
                    dirs_exist_ok=True)

# ── Zip everything ──
shutil.make_archive('/kaggle/working/CS5182_results', 'zip',
                    '/kaggle/working/submission')
print("Packed: /kaggle/working/CS5182_results.zip ✅")
print("Size:", round(os.path.getsize('/kaggle/working/CS5182_results.zip')/1024/1024, 1), "MB")

Packed: /kaggle/working/CS5182_results.zip ✅
Size: 70.0 MB


In [29]:
# Save to Kaggle Dataset (persistent across sessions)
os.makedirs('/kaggle/working/dataset_output', exist_ok=True)
shutil.copy('/kaggle/working/CS5182_results.zip',
            '/kaggle/working/dataset_output/CS5182_results.zip')

'/kaggle/working/dataset_output/CS5182_results.zip'